# Coordinate geometry practice: the math and the Lean proof

A short proof on paper and a short Lean proof can look quite different.
Let's work through six small questions and see where the steps line up.
Each practice starts with the mathematical argument, then reads the matching
Lean file and asks Lean to check it.

The last practice tries an inequality that is too strong. We'll look at both
the error message and a case that shows why the claim fails. That distinction
matters: an unsuccessful argument isn't automatically a false statement.

Run `lake build` from the repository root first, then run the cells in order
with the **Python 3 (ipykernel)** kernel. The [README](../README.md) walks
through setup if you need it. Saved outputs show an actual run; rerunning the
cells will check your current files.

Our points live in $\mathbb{R}^2$. We write $D(A,B)$ for squared distance:

$$
D(A,B)=(b_x-a_x)^2+(b_y-a_y)^2.
$$

When a check passes, Lean has accepted the proof for every point covered by
the statement and its assumptions. It hasn't just tried a few numerical examples.

In [1]:
from pathlib import Path
import sys

# Jupyter may start in the repository root or in notebooks/.
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "lean-toolchain").is_file() and (p / "lakefile.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the euclidean-proof repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.lean_runner import read_lean, check_lean
from IPython.display import Markdown, display
results = {}
print("Ready to check the saved Lean solutions.")

Ready to check the saved Lean solutions.


## Shared Lean definitions

`Point` is a pair of reals. `midpoint` uses real division and is marked `noncomputable`; it remains usable in proofs. The next cell reads these definitions from disk.

In [2]:
display(Markdown("```lean\n" + read_lean("EuclideanProof/Basic.lean", ROOT) + "\n```"))

```lean
import Mathlib.Data.Real.Basic
import Mathlib.Tactic.Ring

/-!
# Our coordinate plane

An ordered pair of real numbers is a point. `P.1` is its x-coordinate;
`P.2` is its y-coordinate. We use squared distance to avoid square roots.
These are coordinate definitions, not a synthetic geometry axiom system.
-/
namespace EuclideanProof

abbrev Point := ℝ × ℝ

-- Division on arbitrary reals is noncomputable: this is a mathematical
-- definition for proofs, not an executable numerical approximation.
noncomputable def midpoint (A B : Point) : Point :=
  ((A.1 + B.1) / 2, (A.2 + B.2) / 2)

def sqDist (A B : Point) : ℝ :=
  (B.1 - A.1) ^ 2 + (B.2 - A.2) ^ 2

end EuclideanProof

```

## Practice 1: Reversing a segment

Reversing a segment is familiar on a drawing: the arrow points the other way. Here we'll write that observation in coordinates and see how Lean checks the two coordinates separately.

[Full practice and exercises](../data/01_segment_reversal.md)

### Question

Let $A=(a_x,a_y)$ and $B=(b_x,b_y)$ be arbitrary points in $\mathbb{R}^2$.
Prove that reversing the segment negates its displacement:

$$
B-A=-(A-B).
$$

### Mathematical proof

For the first coordinate, expand the negation:

$$
-(a_x-b_x)=-a_x+b_x=b_x-a_x.
$$
Likewise, for the second coordinate,

$$
-(a_y-b_y)=b_y-a_y.
$$
Therefore the two coordinate pairs agree:

$$
\begin{aligned}
B-A&=(b_x-a_x,\,b_y-a_y)\\
&=\bigl(-(a_x-b_x),\,-(a_y-b_y)\bigr)\\
&=-(A-B).
\end{aligned}
$$

### Correspondence with Lean

| Mathematical reasoning | Lean code and its role |
| --- | --- |
| Let A and B be arbitrary real coordinate pairs. | `(A B : Point)` |
| To show two pairs are equal, compare their x- and y-coordinates. | `apply Prod.ext` |
| For x: B.x − A.x = −(A.x − B.x), by expanding the minus sign. | First `·` goal, solved by `ring` |
| For y: B.y − A.y = −(A.y − B.y), by the same calculation. | Second `·` goal, solved by `ring` |

`(A B : Point)` means the result holds for every pair of points. `:= by` starts a tactic proof. `apply Prod.ext` turns one equality of pairs into two coordinate goals. Each `·` starts a bullet for one goal. `ring` proves a polynomial identity over the real numbers.


In [3]:
LEAN_FILE = "EuclideanProof/SegmentReversal.lean"
display(Markdown("```lean\n" + read_lean(LEAN_FILE, ROOT) + "\n```"))

```lean
import EuclideanProof.Basic

namespace EuclideanProof

/-- Reversing a segment negates each coordinate displacement. -/
theorem segment_reversal (A B : Point) :
    (B.1 - A.1, B.2 - A.2) = (-(A.1 - B.1), -(A.2 - B.2)) := by
  -- Equality of pairs reduces to equality of their two coordinates.
  apply Prod.ext
  · -- First goal: B.x - A.x = -(A.x - B.x).
    ring
  · -- Second goal: the same calculation for y.
    ring

end EuclideanProof

```

In [4]:
print(f"$ lake env lean {LEAN_FILE}")
result = check_lean(LEAN_FILE, ROOT)
print(f"Exit code: {result.returncode}")
results[LEAN_FILE] = result.returncode

$ lake env lean EuclideanProof/SegmentReversal.lean


PASS: EuclideanProof/SegmentReversal.lean
Exit code: 0


## Practice 2: The midpoint is independent of endpoint order

A midpoint shouldn't depend on which endpoint we name first. This is a small example, but it's a useful place to see how an ordinary fact about addition becomes a Lean proof.

[Full practice and exercises](../data/02_midpoint_symmetry.md)

### Question

For $A=(a_x,a_y)$ and $B=(b_x,b_y)$, define their midpoint by

$$
M(A,B)=\left(\frac{a_x+b_x}{2},\frac{a_y+b_y}{2}\right).
$$
Prove that $M(A,B)=M(B,A)$.

### Mathematical proof

By commutativity of real addition,

$$
a_x+b_x=b_x+a_x,\qquad a_y+b_y=b_y+a_y.
$$
Substituting these equalities in the midpoint formula gives

$$
\begin{aligned}
M(A,B)
&=\left(\frac{a_x+b_x}{2},\frac{a_y+b_y}{2}\right)\\
&=\left(\frac{b_x+a_x}{2},\frac{b_y+a_y}{2}\right)\\
&=M(B,A).
\end{aligned}
$$

### Correspondence with Lean

| Mathematical reasoning | Lean code and its role |
| --- | --- |
| Write each midpoint as the pair of coordinate averages. | `midpoint` in `simp only [midpoint, add_comm]` unfolds the definition. |
| Use A.x + B.x = B.x + A.x, and likewise for y. | `add_comm` supplies commutativity of addition. |
| The equal coordinate averages give equal midpoint pairs. | `simp only` completes both coordinate simplifications and the pair equality. |

`midpoint` is a definition in Basic.lean. `simp only [midpoint, add_comm]` unfolds that definition and simplifies with the commutativity law for addition. The word `only` limits the simplification rules to the ones listed (plus basic computation). `simp` handles the pair components too.


In [5]:
LEAN_FILE = "EuclideanProof/MidpointSymmetry.lean"
display(Markdown("```lean\n" + read_lean(LEAN_FILE, ROOT) + "\n```"))

```lean
import EuclideanProof.Basic

namespace EuclideanProof

/-- A segment's midpoint does not depend on which endpoint is named first. -/
theorem midpoint_symmetry (A B : Point) : midpoint A B = midpoint B A := by
  -- Unfold our definition, then use commutativity of addition in each coordinate.
  simp only [midpoint, add_comm]

end EuclideanProof

```

In [6]:
print(f"$ lake env lean {LEAN_FILE}")
result = check_lean(LEAN_FILE, ROOT)
print(f"Exit code: {result.returncode}")
results[LEAN_FILE] = result.returncode

$ lake env lean EuclideanProof/MidpointSymmetry.lean


PASS: EuclideanProof/MidpointSymmetry.lean
Exit code: 0


## Practice 3: Squared distance is symmetric

Swapping two points changes the signs of their coordinate differences, but it doesn't change their distance. Let's follow what happens when we square those differences, then compare the algebra with Lean's proof.

[Full practice and exercises](../data/03_distance_symmetry.md)

### Question

For $A=(a_x,a_y)$ and $B=(b_x,b_y)$, define squared distance by

$$
D(A,B)=(b_x-a_x)^2+(b_y-a_y)^2.
$$
Prove that $D(A,B)=D(B,A)$. The function $D$ is named `sqDist` in Lean.

### Mathematical proof

Each coordinate difference changes sign when the endpoints are reversed:

$$
b_x-a_x=-(a_x-b_x),\qquad b_y-a_y=-(a_y-b_y).
$$
Since $(-t)^2=t^2$ for every real number $t$,

$$
\begin{aligned}
D(A,B)
&=(b_x-a_x)^2+(b_y-a_y)^2\\
&=\bigl(-(a_x-b_x)\bigr)^2+\bigl(-(a_y-b_y)\bigr)^2\\
&=(a_x-b_x)^2+(a_y-b_y)^2\\
&=D(B,A).
\end{aligned}
$$

### Correspondence with Lean

| Mathematical reasoning | Lean code and its role |
| --- | --- |
| Expand both squared distances into sums of squared coordinate differences. | `dsimp [sqDist]` |
| Reversing each subtraction changes its sign; squaring removes that sign. | `ring` checks this algebraic identity for both coordinates. |
| The two sums of squared differences are equal. | The same `ring` closes the equality goal; it does not call the segment-reversal theorem. |

`dsimp [sqDist]` unfolds our definition. It leaves a polynomial equality. `ring` normalizes each side and proves that they agree. This proof checks the same algebra as the informal argument, without invoking the earlier segment theorem.


In [7]:
LEAN_FILE = "EuclideanProof/DistanceSymmetry.lean"
display(Markdown("```lean\n" + read_lean(LEAN_FILE, ROOT) + "\n```"))

```lean
import EuclideanProof.Basic

namespace EuclideanProof

/-- Squaring removes the sign change from reversing a segment. -/
theorem sqDist_symmetry (A B : Point) : sqDist A B = sqDist B A := by
  -- `dsimp` unfolds the definition, exposing a polynomial identity.
  dsimp [sqDist]
  -- `ring` checks equality by putting both polynomials in a normal form.
  ring

end EuclideanProof

```

In [8]:
print(f"$ lake env lean {LEAN_FILE}")
result = check_lean(LEAN_FILE, ROOT)
print(f"Exit code: {result.returncode}")
results[LEAN_FILE] = result.returncode

$ lake env lean EuclideanProof/DistanceSymmetry.lean


PASS: EuclideanProof/DistanceSymmetry.lean
Exit code: 0


## Practice 4: Squared distance along a horizontal line

When two points have the same height, the vertical part of their distance disappears. The interesting part here is seeing exactly where Lean uses that assumption.

[Full practice and exercises](../data/04_horizontal_distance.md)

### Question

Let $A=(a_x,a_y)$ and $B=(b_x,b_y)$ satisfy $a_y=b_y$.
Using the squared-distance formula

$$
D(A,B)=(b_x-a_x)^2+(b_y-a_y)^2,
$$
prove that $D(A,B)=(b_x-a_x)^2$. The function $D$ is named `sqDist` in Lean.

### Mathematical proof

The hypothesis $a_y=b_y$ implies

$$
b_y-a_y=b_y-b_y=0.
$$
Substitute this into the squared-distance formula:

$$
\begin{aligned}
D(A,B)
&=(b_x-a_x)^2+(b_y-a_y)^2\\
&=(b_x-a_x)^2+(b_y-b_y)^2\\
&=(b_x-a_x)^2+0^2\\
&=(b_x-a_x)^2.
\end{aligned}
$$
The assumption is used in the second line; the remaining steps are algebra.

### Correspondence with Lean

| Mathematical reasoning | Lean code and its role |
| --- | --- |
| Assume the two y-coordinates are equal. | `(h : A.2 = B.2)` |
| Write sqDist(A,B) = (B.x − A.x)² + (B.y − A.y)². | `unfold sqDist` |
| Substitute B.y for A.y using the assumption. | `rw [h]` |
| Since B.y − B.y = 0, its square vanishes and adding zero changes nothing. | `simp` |

`(h : A.2 = B.2)` names the hypothesis h. `unfold sqDist` exposes its formula. `rw [h]` rewrites A.y as B.y. `simp` uses elementary rules such as t − t = 0, 0² = 0, and t + 0 = t. A proof establishes the conclusion under its stated assumptions; it does not establish the assumptions themselves.


In [9]:
LEAN_FILE = "EuclideanProof/HorizontalDistance.lean"
display(Markdown("```lean\n" + read_lean(LEAN_FILE, ROOT) + "\n```"))

```lean
import EuclideanProof.Basic

namespace EuclideanProof

/-- Equal y-coordinates make the vertical contribution to squared distance zero. -/
theorem horizontal_sqDist (A B : Point) (h : A.2 = B.2) :
    sqDist A B = (B.1 - A.1) ^ 2 := by
  -- The named hypothesis `h` is a premise, not something we prove here.
  unfold sqDist
  -- Replace A.y with B.y using h.
  rw [h]
  -- B.y - B.y = 0, 0² = 0, and t + 0 = t.
  simp

end EuclideanProof

```

In [10]:
print(f"$ lake env lean {LEAN_FILE}")
result = check_lean(LEAN_FILE, ROOT)
print(f"Exit code: {result.returncode}")
results[LEAN_FILE] = result.returncode

$ lake env lean EuclideanProof/HorizontalDistance.lean


PASS: EuclideanProof/HorizontalDistance.lean
Exit code: 0


## Practice 5: A coordinate Pythagorean identity

The familiar Pythagorean identity is a good place to compare a short calculation with a short Lean proof. We'll use a triangle whose legs follow the coordinate axes, so we can see every term directly.

[Full practice and exercises](../data/05_pythagorean.md)

### Question

Let $O=(0,0)$, $H=(x,0)$, and $P=(x,y)$, where $x,y\in\mathbb{R}$.
With $D$ denoting squared distance (`sqDist` in Lean), prove

$$
D(O,P)=D(O,H)+D(H,P).
$$

### Mathematical proof

Compute the three squared distances directly:

$$
\begin{aligned}
D(O,P)&=(x-0)^2+(y-0)^2=x^2+y^2,\\
D(O,H)&=(x-0)^2+(0-0)^2=x^2,\\
D(H,P)&=(x-x)^2+(y-0)^2=y^2.
\end{aligned}
$$
Consequently,

$$
D(O,P)=x^2+y^2=D(O,H)+D(H,P).
$$
The legs are parallel to the coordinate axes. This proves the identity for
these coordinates; it also holds when $x=0$ or $y=0$, when the triangle is degenerate.

### Correspondence with Lean

| Mathematical reasoning | Lean code and its role |
| --- | --- |
| Let x and y be arbitrary real numbers, with O=(0,0), H=(x,0), P=(x,y). | `(x y : ℝ)` and the explicit coordinate pairs in the statement |
| Expand OP², OH², and HP² using the squared-distance formula. | `dsimp [sqDist]` |
| Compute OP²=x²+y², OH²=x², HP²=y²; hence OP²=OH²+HP². | `ring` simplifies the coordinate differences and verifies the resulting identity. |

`(x y : ℝ)` universally quantifies over two real coordinates. `dsimp [sqDist]` unfolds the formula and computes the coordinates of the explicitly given pairs. `ring` finishes the algebra. Negative coordinates work too. If x or y is zero, this is a degenerate triangle; the identity remains true.


In [11]:
LEAN_FILE = "EuclideanProof/Pythagorean.lean"
display(Markdown("```lean\n" + read_lean(LEAN_FILE, ROOT) + "\n```"))

```lean
import EuclideanProof.Basic

namespace EuclideanProof

/-- A right triangle with legs parallel to the coordinate axes. -/
theorem pythagorean_axis_aligned (x y : ℝ) :
    sqDist (0, 0) (x, y) =
      sqDist (0, 0) (x, 0) + sqDist (x, 0) (x, y) := by
  -- Expand squared distances and evaluate pair coordinates.
  dsimp [sqDist]
  -- The resulting identity is x² + y² = x² + y².
  ring

end EuclideanProof

```

In [12]:
print(f"$ lake env lean {LEAN_FILE}")
result = check_lean(LEAN_FILE, ROOT)
print(f"Exit code: {result.returncode}")
results[LEAN_FILE] = result.returncode

$ lake env lean EuclideanProof/Pythagorean.lean


PASS: EuclideanProof/Pythagorean.lean
Exit code: 0


## Practice 6: A true inequality and a false stronger claim

A small change from “at least zero” to “greater than zero” makes a real difference. Let's prove the first claim, try the stronger one, and look at why Lean asks us for more.

[Full practice](../data/06_nonnegative_distance.md)

### Question

Let $A=(a_x,a_y)$ and $B=(b_x,b_y)$ be arbitrary points in $\mathbb{R}^2$.
With squared distance defined by

$$
D(A,B)=(b_x-a_x)^2+(b_y-a_y)^2,
$$
prove the inequality $D(A,B)\ge 0$. Can we strengthen it to $D(A,B)>0$
without adding an assumption?

### Mathematical proof

Every real square is nonnegative:

$$
(b_x-a_x)^2\ge 0,\qquad (b_y-a_y)^2\ge 0.
$$
Adding these inequalities yields

$$
D(A,B)=(b_x-a_x)^2+(b_y-a_y)^2\ge 0+0=0.
$$
In Lean, we write this conclusion as `0 ≤ sqDist A B`. Unlike `ring`, which
checks polynomial equalities, this proof uses order lemmas: `sq_nonneg` and
`add_nonneg`. `exact` supplies a proof whose type must match the current goal.

### Correspondence with Lean

| Mathematical reasoning | Lean code and its role |
| --- | --- |
| Expand the squared-distance formula. | `unfold sqDist` |
| Each real square is nonnegative. | `sq_nonneg (B.1 - A.1)` and `sq_nonneg (B.2 - A.2)` |
| A sum of two nonnegative quantities is nonnegative. | `add_nonneg` combines the two proofs; `exact` closes the goal. |
| Coincident points have squared distance zero. | `simp [sqDist]` proves `sqDist_self`. |
| A universal strict inequality would apply to the origin paired with itself. | `intro h` followed by `h (0, 0) (0, 0)` |
| The resulting claim that zero is strictly less than zero is impossible. | `simp [sqDist] at impossible` proves the negation. |

In [13]:
LEAN_FILE = "EuclideanProof/NonnegativeDistance.lean"
display(Markdown("```lean\n" + read_lean(LEAN_FILE, ROOT) + "\n```"))

```lean
import EuclideanProof.Basic

namespace EuclideanProof

/-- Squared distance is a sum of two nonnegative squares. -/
theorem sqDist_nonneg (A B : Point) : 0 ≤ sqDist A B := by
  unfold sqDist
  -- `sq_nonneg t` proves 0 ≤ t²; `add_nonneg` adds two such inequalities.
  exact add_nonneg (sq_nonneg (B.1 - A.1)) (sq_nonneg (B.2 - A.2))

/-- Coincident endpoints give a counterexample to unconditional positivity. -/
theorem sqDist_self (A : Point) : sqDist A A = 0 := by
  simp [sqDist]

/-- Formally refute the assertion that every squared distance is positive. -/
theorem not_always_sqDist_pos : ¬ (∀ A B : Point, 0 < sqDist A B) := by
  -- Suppose the universal strict inequality were true.
  intro h
  -- It would hold for the origin paired with itself.
  have impossible := h (0, 0) (0, 0)
  -- But its squared distance is zero, so this would assert 0 < 0.
  simp [sqDist] at impossible

end EuclideanProof

```

In [14]:
print(f"$ lake env lean {LEAN_FILE}")
result = check_lean(LEAN_FILE, ROOT)
print(f"Exit code: {result.returncode}")
results[LEAN_FILE] = result.returncode

$ lake env lean EuclideanProof/NonnegativeDistance.lean


PASS: EuclideanProof/NonnegativeDistance.lean
Exit code: 0


### Counterexample to strict positivity

The stronger claim $D(A,B)>0$ for **all** $A,B$ is false.
Take $A=B=(0,0)$. Then

$$
D(A,A)=(0-0)^2+(0-0)^2=0,
$$
so the proposed conclusion would be $0>0$, a contradiction.
Thus

$$
\neg\bigl(\forall A,B\in\mathbb{R}^2,\;D(A,B)>0\bigr).
$$
The Lean theorem `sqDist_self` proves the zero-distance calculation, and
`not_always_sqDist_pos` formally proves this negation. `intro h` assumes the
universal claim; applying `h` to the origin twice gives the contradiction.
A strictly positive result needs a further assumption, such as $A\ne B$.

### Watch Lean reject the invalid attempt

The invalid attempt below passes a proof of $0\le D(A,B)$ to a goal
requiring $0<D(A,B)$. Lean reports a **type mismatch**: the weak inequality
is not a proof of the strict inequality.

Lean rejecting an argument doesn't, by itself, mean the statement is false.
We might just need a different proof of a true statement. Here the separate
counterexample above, also checked in Lean, establishes that the universal
strict inequality is false.

The invalid file is intentionally outside the library imports, so normal
`lake build` checks the valid practice problems and succeeds. The notebook checks this
file separately, displays the compiler diagnostics, and catches the expected
failure. It still raises if the invalid file unexpectedly passes or if the
failure is unrelated to this proof mismatch.

In [15]:
FAILED_FILE = "examples/expected_failure/StrictDistance.lean"
display(Markdown("```lean\n" + read_lean(FAILED_FILE, ROOT) + "\n```"))

```lean
import EuclideanProof.NonnegativeDistance

open EuclideanProof

-- DELIBERATELY INVALID: nonnegative does not imply strictly positive.
-- This file is checked explicitly by the demonstration notebook and is
-- not imported by the library, so `lake build` remains successful.
example (A B : Point) : 0 < sqDist A B := by
  -- This theorem proves 0 ≤ sqDist A B, which does not meet the goal 0 < ... .
  exact sqDist_nonneg A B

```

In [16]:
import subprocess

print(f"$ lake env lean {FAILED_FILE}")
try:
    check_lean(FAILED_FILE, ROOT)
except subprocess.CalledProcessError as error:
    diagnostics = (error.output or "") + (error.stderr or "")
    # A missing import or broken setup must not count as the intended failure.
    assert "sqDist_nonneg" in diagnostics and "expected to have type" in diagnostics, diagnostics
    print(f"Exit code: {error.returncode} (expected nonzero)")
    print("EXPECTED FAILURE: nonnegative does not imply strictly positive.")
    expected_failure_code = error.returncode
else:
    raise AssertionError("The deliberately invalid proof unexpectedly passed.")

$ lake env lean examples/expected_failure/StrictDistance.lean


examples/expected_failure/StrictDistance.lean:10:2: error: Type mismatch
  sqDist_nonneg A B
has type
  0 ≤ sqDist A B
but is expected to have type
  0 < sqDist A B
Exit code: 1 (expected nonzero)
EXPECTED FAILURE: nonnegative does not imply strictly positive.


## Results

The six valid practice files must pass. The intentionally invalid file must fail
with the expected type mismatch. Its caught exception is part of this
exercise; an unexpected failure in a valid proof still stops the notebook.
The table below records both kinds of checker results.

In [17]:
assert len(results) == 6, "Run all six valid practice checks above."
assert expected_failure_code != 0
rows = "\n".join(f"| `{name}` | {code} | PASS |" for name, code in results.items())
rows += f"\n| `{FAILED_FILE}` | {expected_failure_code} | EXPECTED FAILURE |"
display(Markdown("| Lean file | Exit code | Result |\n| --- | --- | --- |\n" + rows))

| Lean file | Exit code | Result |
| --- | --- | --- |
| `EuclideanProof/SegmentReversal.lean` | 0 | PASS |
| `EuclideanProof/MidpointSymmetry.lean` | 0 | PASS |
| `EuclideanProof/DistanceSymmetry.lean` | 0 | PASS |
| `EuclideanProof/HorizontalDistance.lean` | 0 | PASS |
| `EuclideanProof/Pythagorean.lean` | 0 | PASS |
| `EuclideanProof/NonnegativeDistance.lean` | 0 | PASS |
| `examples/expected_failure/StrictDistance.lean` | 1 | EXPECTED FAILURE |